# Medical LLM Notebook 02 — MedGemma Attention Extraction

Extract and validate MedGemma attention for a reproducible factual–hallucinated pilot pair.

The notebook uses `google/medgemma-4b-it`, bfloat16 precision, eager attention, and the model chat template. It stores last-layer, final-four-layer mean, and all-layer mean attention representations. The final-four-layer mean is the primary representation used downstream.

In [ ]:
!pip -q install transformers accelerate huggingface_hub pandas numpy torch

In [ ]:
import os
import gc
import json
import shutil
import numpy as np
import pandas as pd
import torch

from google.colab import files
from huggingface_hub import notebook_login
from transformers import AutoProcessor, AutoModelForImageTextToText

SEED = 42
MODEL_ID = "google/medgemma-4b-it"

torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


## 1. Upload the 100-pair file from Notebook 01

In [ ]:
uploaded = files.upload()
csv_files = [x for x in uploaded if x.lower().endswith(".csv")]
assert csv_files, "Upload MedHallu_Pilot_100_Pairs.csv"

pairs_df = pd.read_csv(csv_files[0])
print("Pairs:", len(pairs_df))
display(pairs_df.head())


## 2. Authenticate and load MedGemma

In [ ]:
notebook_login()


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="eager"
)

model.eval()
tokenizer = processor.tokenizer

print("Model loaded:", MODEL_ID)


## 3. Attention extraction helpers

In [ ]:
def _turn_content_indices(input_ids, tokens, turn_start, turn_end):
    indices = list(range(turn_start + 1, turn_end))

    while indices:
        idx = indices[0]
        decoded = tokenizer.decode(
            [int(input_ids[idx])],
            skip_special_tokens=False
        )
        clean = decoded.strip().lower()

        if clean in {"user", "model", "assistant"} or clean == "":
            indices.pop(0)
        else:
            break

    while indices:
        idx = indices[-1]
        decoded = tokenizer.decode(
            [int(input_ids[idx])],
            skip_special_tokens=False
        )
        if decoded.strip() == "":
            indices.pop()
        else:
            break

    return indices


def locate_question_answer_tokens(input_ids):
    ids = input_ids.tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)

    turn_starts = [
        i for i, token in enumerate(tokens)
        if token == "<start_of_turn>"
    ]
    turn_ends = [
        i for i, token in enumerate(tokens)
        if token == "<end_of_turn>"
    ]

    if len(turn_starts) < 2 or len(turn_ends) < 2:
        raise ValueError("Could not identify user and assistant turns.")

    user_start = turn_starts[0]
    user_end = next(e for e in turn_ends if e > user_start)
    assistant_start = next(s for s in turn_starts if s > user_end)
    assistant_end = next(e for e in turn_ends if e > assistant_start)

    q_idx = _turn_content_indices(
        ids, tokens, user_start, user_end
    )
    a_idx = _turn_content_indices(
        ids, tokens, assistant_start, assistant_end
    )

    return q_idx, a_idx, tokens


In [ ]:
def extract_attention_representations(question, response):
    messages = [
        {
            "role": "user",
            "content": [{"type": "text", "text": str(question)}]
        },
        {
            "role": "assistant",
            "content": [{"type": "text", "text": str(response)}]
        }
    ]

    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    input_ids_cpu = inputs["input_ids"][0].detach().cpu()

    with torch.inference_mode():
        outputs = model(
            **inputs,
            output_attentions=True,
            use_cache=False,
            return_dict=True
        )

    attentions = outputs.attentions
    if attentions is None:
        raise RuntimeError("No attention tensors returned.")

    layer_head_means = [
        a[0].float().mean(dim=0)
        for a in attentions
    ]

    last = layer_head_means[-1].cpu().numpy()
    last4 = torch.stack(layer_head_means[-4:]).mean(dim=0).cpu().numpy()
    all_layers = torch.stack(layer_head_means).mean(dim=0).cpu().numpy()

    for name, A in {
        "last": last,
        "last4": last4,
        "all_layers": all_layers
    }.items():
        if not np.isfinite(A).all():
            raise ValueError(f"NaN/Inf in {name} attention.")

    q_idx, a_idx, tokens = locate_question_answer_tokens(input_ids_cpu)

    result = {
        "last": last,
        "last4": last4,
        "all_layers": all_layers,
        "input_ids": input_ids_cpu.numpy(),
        "tokens": tokens,
        "question_indices": q_idx,
        "answer_indices": a_idx,
        "num_layers": len(attentions),
        "num_heads": attentions[0].shape[1]
    }

    del outputs, attentions, inputs, layer_head_means
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result


## 4. Reproduce pilot pair MEDH_096

In [ ]:
PAIR_ID = "MEDH_096"

row = pairs_df[pairs_df["Pair_ID"] == PAIR_ID].iloc[0]

question = row["Question"]
factual_response = row["Ground_Truth"]
hall_response = row["Hallucinated_Answer"]

factual = extract_attention_representations(
    question,
    factual_response
)

hall = extract_attention_representations(
    question,
    hall_response
)

print("Layers:", factual["num_layers"])
print("Heads:", factual["num_heads"])
print("Factual sequence:", factual["last4"].shape)
print("Hallucinated sequence:", hall["last4"].shape)


In [ ]:
def row_sum_diagnostics(A):
    s = A.sum(axis=1)
    return {
        "min": float(s.min()),
        "max": float(s.max()),
        "mean": float(s.mean())
    }

print("Factual last4 row sums:", row_sum_diagnostics(factual["last4"]))
print("Hall last4 row sums:", row_sum_diagnostics(hall["last4"]))


## 5. Save Notebook 02 outputs

In [ ]:
OUT = "/content/Notebook02_MEDH096_outputs"
os.makedirs(OUT, exist_ok=True)

np.savez_compressed(
    f"{OUT}/MEDH_096_attention_pair.npz",
    factual_last=factual["last"],
    factual_last4=factual["last4"],
    factual_all_layers=factual["all_layers"],
    hallucinated_last=hall["last"],
    hallucinated_last4=hall["last4"],
    hallucinated_all_layers=hall["all_layers"]
)

with open(f"{OUT}/MEDH_096_tokens.json", "w") as f:
    json.dump(
        {
            "factual_tokens": factual["tokens"],
            "hallucinated_tokens": hall["tokens"],
            "factual_input_ids": factual["input_ids"].tolist(),
            "hallucinated_input_ids": hall["input_ids"].tolist(),
            "factual_question_indices": factual["question_indices"],
            "factual_answer_indices": factual["answer_indices"],
            "hallucinated_question_indices": hall["question_indices"],
            "hallucinated_answer_indices": hall["answer_indices"]
        },
        f,
        indent=2
    )

with open(f"{OUT}/MEDH_096_metadata.json", "w") as f:
    json.dump(
        {
            "Pair_ID": PAIR_ID,
            "Difficulty": str(row["Difficulty"]),
            "Hallucination_Category": str(row["Hallucination_Category"]),
            "Model": MODEL_ID,
            "dtype": "bfloat16",
            "attention_representation": "mean across heads, then final four layers",
            "seed": SEED
        },
        f,
        indent=2
    )

archive = shutil.make_archive(
    "/content/Notebook2_MEDH096_outputs",
    "zip",
    OUT
)

print("Created:", archive)
files.download(archive)
